# IDR MVP — Phase 1: Dataset Analysis & Preprocessing

**Objective:** Inspect and preprocess the IO-VNBD benchmark dataset for Intelligent Dead Reckoning (IDR).

### Pipeline Steps:
1. **Load** synchronized smartphone + vehicle dataset pairs
2. **Inspect** fields, determine sampling rates, check data quality
3. **Clean** missing samples, duplicate timestamps, outliers
4. **Filter** accelerometer and gyroscope signals (Butterworth low-pass @ 4 Hz)
5. **Synchronize** multi-rate streams (10 Hz IMU + 1 Hz GNSS + Reference)
6. **Convert** to standard internal IDR format
7. **Visualize** IMU dynamics, GNSS track vs Vehicle Ground Truth

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import (
    DatasetLoader,
    DatasetInspector,
    DataCleaner,
    SignalFilter,
    SensorSynchronizer,
    FormatConverter,
)

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120
print('IDR Preprocessing modules imported successfully!')

## 1. Discover & Load IO-VNBD Dataset

In [ ]:
dataset_root = PROJECT_ROOT / 'data' / 'raw' / 'IO-VNBD'
loader = DatasetLoader(str(dataset_root))
summary = loader.summary()

print(f"Total CSV files:      {summary['total_csv_files']}")
print(f"Smartphone files:     {summary['smartphone_files']}")
print(f"Vehicle files:        {summary['vehicle_files']}")
print(f"Synchronized pairs:   {summary['synced_pairs_count']}")
print(f"Sample Trip IDs:      {summary['synced_pair_trips'][:10]}")

## 2. Load Synchronized Trip Pair (Smartphone + Vehicle Ground Truth)

In [ ]:
# Load trip 's1' with both Smartphone sensors and Vehicle CAN/GNSS reference ground truth
df_raw, meta = loader.load_synced_pair(trip_name='s1')
print(f"Trip:       {meta['trip_id']}")
print(f"Raw Shape:  {df_raw.shape}")
print(f"Rows:       {meta['merged_rows']:,}")
df_raw.head(3)

## 3. Inspect Dataset & Compute Sampling Rates

In [ ]:
inspector = DatasetInspector(df_raw, name=meta['trip_id'])
inspector.print_summary()

identified = inspector.identify_columns()
sampling_info = inspector.compute_sampling_rate()
print(f"\nDetected Sampling Rate: {sampling_info['sampling_rate_hz']:.2f} Hz")
print(f"Total Duration:         {sampling_info['duration_seconds']:.1f} s ({sampling_info['duration_seconds']/60:.1f} min)")

## 4. Clean Data (Outlier Detection & Interpolation)

In [ ]:
time_col = identified['timestamp'][0]
accel_cols = identified['accelerometer']
gyro_cols = identified['gyroscope']
sensor_cols = accel_cols + gyro_cols

cleaner = DataCleaner(df_raw)
df_cleaned = cleaner.clean(
    time_col=time_col,
    sensor_columns=sensor_cols,
    interpolate_max_gap=5,
    outlier_z_threshold=5.0,
    gap_threshold_seconds=1.0,
)

## 5. Filter IMU Signals (Butterworth Low-pass @ 4 Hz)

In [ ]:
fs = sampling_info['sampling_rate_hz']
sig_filter = SignalFilter(sampling_rate_hz=fs)

df_filtered = sig_filter.filter_accelerometer(df_cleaned, accel_cols, cutoff_hz=4.0)
df_filtered = sig_filter.filter_gyroscope(df_filtered, gyro_cols, cutoff_hz=4.0)

# Plot Filter Comparison on 1000 samples
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle('Accelerometer Signals: Raw vs Butterworth Low-pass (4 Hz)', fontsize=14, fontweight='bold')
for i, col in enumerate(accel_cols[:3]):
    axes[i].plot(df_raw[col].values[:1000], label='Raw', color='gray', alpha=0.6, linewidth=0.8)
    axes[i].plot(df_filtered[col].values[:1000], label='Filtered (4 Hz)', color='#007acc', linewidth=1.5)
    axes[i].set_ylabel(col, fontsize=10)
    axes[i].legend(loc='upper right')
axes[-1].set_xlabel('Sample Index')
plt.tight_layout()
plt.show()

## 6. Synchronize Streams & Convert to Standard Format

In [ ]:
synchronizer = SensorSynchronizer(target_rate_hz=fs)
df_synced = synchronizer.synchronize(
    df_filtered,
    time_col=time_col,
    gnss_columns=identified['gnss'],
    gnss_strategy='forward_fill',
)

converter = FormatConverter()
df_standard = converter.convert(
    df_synced,
    identified_sensors=identified,
    keep_extra=True,
)

print('Standard format columns:', list(df_standard.columns[:15]))
df_standard[['timestamp', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'gnss_lat', 'gnss_lon', 'gnss_speed', 'ref_lat', 'ref_lon', 'ref_speed']].head()

## 7. Trajectory Visualization: Smartphone GNSS vs Vehicle Ground Truth

In [ ]:
plt.figure(figsize=(12, 10))
gnss_valid = df_standard[['gnss_lat', 'gnss_lon']].dropna()
ref_valid = df_standard[['ref_lat', 'ref_lon']].dropna()

if len(gnss_valid) > 0:
    plt.scatter(gnss_valid['gnss_lon'], gnss_valid['gnss_lat'], c='blue', s=3, alpha=0.5, label='Smartphone GNSS (1 Hz)')
if len(ref_valid) > 0:
    plt.plot(ref_valid['ref_lon'], ref_valid['ref_lat'], color='red', linewidth=1.5, alpha=0.8, label='Vehicle Ground Truth (10 Hz)')

plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.title(f'Trajectory: Smartphone GNSS vs Vehicle Ground Truth ({meta["trip_id"]})', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()